# Demo - Model Routing From Measured Repository Signals
**Day 1 - Session 1, Topic 2**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kpassoubady/agent-orchestration-companion/blob/main/day1/demos-notebook/demo-routing-signals-measured.ipynb)

**Goal:** Compute blast radius, verification cover, and reversibility from the real repository so the routing tier is an output of measurement, not an opinion.

Scoring a task by hand makes the score the answer. Here every signal is measured: blast radius counts real importers, verification cover counts real test methods, and reversibility is read from the schema contract on disk. Three candidates land in three tiers because the repository differs, not because the tiers were typed in.

> Model tier names are dated examples. The measured signals are the durable rule.


## 1. Setup

Locate the course files and put `demo_support` on the import path.


In [5]:
# Setup: make the course files and demo_support importable.
# On Colab nothing is present yet, so clone the companion repo once.
# Locally this finds your existing checkout and clones nothing.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kpassoubady/agent-orchestration-companion.git"
MARKER = Path("lab-workspace-solution") / "router.py"


def find_repo_root():
    directory = Path.cwd()
    for _ in range(6):
        if (directory / MARKER).exists():
            return directory
        directory = directory.parent
    clone = Path.cwd() / "agent-orchestration-companion"
    if not (clone / MARKER).exists():
        print(f"Cloning {{REPO_URL}} ...")
        subprocess.run(
            ["git", "clone", "--depth", "1", "-q", REPO_URL, str(clone)],
            check=True,
            env=dict(os.environ, GIT_TERMINAL_PROMPT="0"),
        )
    return clone


ROOT = find_repo_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "day1" / "demos"))

# Colab has no global Git identity; the demos set a local one per sandbox repo.
print("Course root:", ROOT)
print("Git:", subprocess.run(["git", "--version"], capture_output=True, text=True).stdout.strip())

Course root: /Users/kangs/code/github/agent-orchestration-companion
Git: git version 2.54.0 (Apple Git-157)


## 2. Define the measurement functions

Each signal is computed from the repository. Nothing here stores a verdict.


In [6]:
import ast
import json
from pathlib import Path

from demo_support import assert_true, heading, sandbox, show_evidence

CANDIDATES = [
    {"task": "Reword the shipment email subject line",
     "target": Path("channels") / "email.py", "touches_contract": False},
    {"task": "Add a task-card note for the SMS owner",
     "target": Path("task-cards") / "sms.md", "touches_contract": False},
    {"task": "Change the locked shipment event contract",
     "target": Path("schemas") / "shipment-event.json", "touches_contract": True},
]


def imports_target(source, target):
    dotted = str(target.with_suffix("")).replace("/", ".")
    for node in ast.walk(ast.parse(source)):
        names = []
        if isinstance(node, ast.ImportFrom) and node.module:
            names = [node.module]
        elif isinstance(node, ast.Import):
            names = [alias.name for alias in node.names]
        if any(name == dotted or name.startswith(f"{dotted}.") for name in names):
            return True
    return False


def importers_of(root, target):
    """Production modules that import the target. Tests verify, so they are excluded."""
    if target.suffix != ".py":
        return []
    found = []
    for path in root.rglob("*.py"):
        if "__pycache__" in path.parts or path == root / target or path.name.startswith("test_"):
            continue
        if imports_target(path.read_text(), target):
            found.append(str(path.relative_to(root)))
    return sorted(set(found))


def contract_consumers(root, target):
    return sorted(
        str(path.relative_to(root))
        for path in root.rglob("*.py")
        if "__pycache__" not in path.parts and target.name in path.read_text()
    )


def test_methods_covering(root, target):
    covering = []
    for path in sorted((root / "tests").rglob("test_*.py")):
        source = path.read_text()
        direct = target.suffix == ".py" and imports_target(source, target)
        indirect = target.suffix != ".py" and target.name in source
        if not (direct or indirect):
            continue
        for node in ast.walk(ast.parse(source)):
            if isinstance(node, ast.FunctionDef) and node.name.startswith("test_"):
                covering.append(f"{path.relative_to(root)}::{node.name}")
    return sorted(covering)


print("Measurement functions defined. Nothing scored yet.")

Measurement functions defined. Nothing scored yet.


## 3. Measure every candidate against the real repository

Watch each signal come from a file on disk: imports, test methods, and the schema.


In [7]:
def reversibility(root, candidate):
    if not candidate["touches_contract"]:
        return True, "module-local change, revert by restoring the file"
    schema = json.loads((root / "schemas" / "shipment-event.json").read_text())
    locked = schema.get("additionalProperties") is False
    return (not locked,
            f"schema version {schema['version']} locks "
            f"additionalProperties={schema.get('additionalProperties')}")


def tier(blast, cover, reversible):
    """One rule, applied to measured signals only."""
    if not reversible:
        return "Strong model, high effort, human approval required"
    if blast >= 2 or cover >= 5:
        return "Strong model, high effort, risk-based approval"
    if cover >= 3:
        return "Mid model, medium effort, focused check"
    return "Efficient model, low effort, no approval"


sandbox_context = sandbox(with_git=False)
WORK, _ = sandbox_context.__enter__()

RESULTS = []
for candidate in CANDIDATES:
    target = candidate["target"]
    heading(f"Candidate: {candidate['task']}")
    show_evidence("target file", str(target))

    if target.suffix == ".py":
        reached, label = importers_of(WORK, target), "modules importing it"
    else:
        reached, label = contract_consumers(WORK, target), "modules reading it"
    cover = test_methods_covering(WORK, target)
    reversible, why = reversibility(WORK, candidate)

    show_evidence(label, ", ".join(reached) or "(none)")
    show_evidence("blast radius (measured)", len(reached))
    show_evidence("test methods covering", len(cover))
    for name in cover:
        show_evidence("", name)
    show_evidence("reversible", "yes" if reversible else "NO")
    show_evidence("evidence", why)

    decision = tier(len(reached), len(cover), reversible)
    show_evidence("routing tier", decision)
    RESULTS.append((candidate["task"], len(reached), len(cover), reversible, decision))


Candidate: Reword the shipment email subject line
-------------------------------------------------
  target file                channels/email.py
  modules importing it       router.py
  blast radius (measured)    1
  test methods covering      3
                             tests/test_email.py::test_renders_subject_and_body
                             tests/test_security.py::test_email_escapes_untrusted_html
                             tests/test_security.py::test_sms_normalizes_control_whitespace
  reversible                 yes
  evidence                   module-local change, revert by restoring the file
  routing tier               Mid model, medium effort, focused check

Candidate: Add a task-card note for the SMS owner
-------------------------------------------------
  target file                task-cards/sms.md
  modules reading it         (none)
  blast radius (measured)    0
  test methods covering      0
  reversible                 yes
  evidence                   mod

## 4. Compare the tiers and verify the evidence

One rule, three measured inputs, three different answers.


In [8]:
heading("Side-by-side: one rule, different measured inputs")
for task, blast, cover, reversible, decision in RESULTS:
    show_evidence(task[:24], f"blast={blast} cover={cover} reversible={reversible} -> {decision.split(',')[0]}")

heading("Evidence checks")
assert_true(RESULTS[0][1] >= 1, "the email change's blast radius came from real imports")
assert_true(any("Efficient" in result[4] for result in RESULTS),
            "a change nothing imports stayed in the cheap tier")
assert_true(RESULTS[0][2] >= 1, "real test methods were counted, not estimated")
assert_true(any(result[3] is False for result in RESULTS),
            "the schema file on disk proved one change irreversible")
assert_true(len({result[4] for result in RESULTS}) >= 3,
            "measured signals produced three distinct tiers")

sandbox_context.__exit__(None, None, None)
print("\nTakeaway: Measure blast radius, verification cover, and reversibility")
print("from the repository, then let one rule assign the tier.")


Side-by-side: one rule, different measured inputs
-------------------------------------------------
  Reword the shipment emai   blast=1 cover=3 reversible=True -> Mid model
  Add a task-card note for   blast=0 cover=0 reversible=True -> Efficient model
  Change the locked shipme   blast=2 cover=0 reversible=False -> Strong model

Evidence checks
---------------
  [verified] the email change's blast radius came from real imports
  [verified] a change nothing imports stayed in the cheap tier
  [verified] real test methods were counted, not estimated
  [verified] the schema file on disk proved one change irreversible
  [verified] measured signals produced three distinct tiers

Takeaway: Measure blast radius, verification cover, and reversibility
from the repository, then let one rule assign the tier.


### Expected output

- Email renderer: blast radius 1 (`router.py`), 3 covering test methods,
  reversible, so the **Mid model** tier.
- Task-card note: blast radius 0, no tests, reversible, so the
  **Efficient model** tier.
- Locked schema: read from disk as `additionalProperties=False`, so
  **Strong model with human approval required**.
- Five `[verified]` lines confirming three distinct tiers from measured inputs.
